# EDA and Preprocessing for PubMed and BC5CDR

This notebook performs exploratory data analysis and preprocessing for the biomedical NER final project.

Two datasets are used in this stage:

1. **PubMed scraped corpus**  
   This dataset contains biomedical article titles and abstracts collected from PubMed. It is treated as an unlabeled corpus and will later receive pseudo-labels through distant supervision.

2. **BC5CDR**  
   This dataset contains token-level gold annotations for biomedical named entities, specifically `Chemical` and `Disease`.

The main goals of this notebook are:

1. Inspect the structure and quality of the PubMed scraped dataset.
2. Clean and prepare the PubMed corpus for sentence-level NER processing.
3. Load and inspect the BC5CDR dataset.
4. Convert BC5CDR labels into readable BIO tags.
5. Apply lightweight preprocessing decisions to BC5CDR.
6. Save both datasets in a consistent format for downstream distant supervision and NER training.

The output of this notebook will be used in the next stage of the project: distant supervision for `Gene` and `Virus` pseudo-labeling.

## Import Libraries

This section imports the required libraries for data loading, exploratory analysis, text preprocessing, visualization, and saving processed datasets.

The notebook uses `pandas` for tabular data processing, `re` for regular-expression-based text processing, `matplotlib` for simple visual inspection, and `datasets` to load the BC5CDR dataset from Hugging Face.

In [1]:
import os
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

d:\AI_Journey\NLP\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

This section defines input and output paths used throughout the notebook.

The PubMed input file is expected to come from the previous scraping notebook. The output files generated here will be used by the distant supervision notebook.

Using a configuration section keeps the notebook easier to maintain and avoids hardcoding paths across multiple cells.

In [ ]:
# Input path from the PubMed scraping notebook
PUBMED_INPUT_PATH = "pubmed_abstracts_2022_2026_clean_with_text.csv"

# Output directory for preprocessed datasets
OUTPUT_DIR = "preprocessed_data"
PUBMED_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "pubmed")
BC5CDR_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "bc5cdr")

os.makedirs(PUBMED_OUTPUT_DIR, exist_ok=True)
os.makedirs(BC5CDR_OUTPUT_DIR, exist_ok=True)

# Output paths
PUBMED_SENTENCE_JSONL_PATH = os.path.join(
    PUBMED_OUTPUT_DIR,
    "pubmed_2022_2026_sentence_level_preprocessed.jsonl"
)

PUBMED_SENTENCE_CSV_PATH = os.path.join(
    PUBMED_OUTPUT_DIR,
    "pubmed_2022_2026_sentence_level_preprocessed.csv"
)

# Basic preprocessing parameters
MIN_PUBMED_SENTENCE_TOKENS = 4
MIN_BC5CDR_SHORT_SENTENCE_TOKENS = 3

print("PubMed input path:", PUBMED_INPUT_PATH)
print("PubMed output directory:", PUBMED_OUTPUT_DIR)
print("BC5CDR output directory:", BC5CDR_OUTPUT_DIR)

## Load PubMed Scraped Dataset

This section loads the cleaned PubMed dataset generated from the scraping notebook.

At this stage, the PubMed corpus is still document-level, where each row represents one PubMed article containing metadata, title, abstract, and a combined `text` column.

The combined `text` column will later be split into sentences and tokenized for NER-style preprocessing.

In [ ]:
pubmed_df = pd.read_csv(PUBMED_INPUT_PATH)

print("PubMed dataset shape:", pubmed_df.shape)
pubmed_df.head()

## Inspect PubMed Columns and Missing Values

This section checks the available columns and missing values in the PubMed dataset.

Missing values are important to inspect before preprocessing because the downstream NER pipeline requires valid text data. Rows without usable title, abstract, or combined text should not be included in sentence-level preprocessing.

In [ ]:
print("Columns:")
print(pubmed_df.columns.tolist())

print("\nMissing values:")
print(pubmed_df.isna().sum())

## Validate Required PubMed Columns

This section validates whether the required columns exist in the PubMed dataset.

The preprocessing pipeline expects article metadata and text fields such as `pmid`, `title`, `abstract`, `publication_year`, `journal`, `url`, and `text`.

If any required column is missing, the notebook will raise an error so the issue can be fixed before continuing.

In [ ]:
required_pubmed_columns = [
    "pmid",
    "title",
    "abstract",
    "publication_year",
    "journal",
    "url",
    "text"
]

missing_required_columns = [
    column for column in required_pubmed_columns
    if column not in pubmed_df.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print("All required PubMed columns are available.")

## Remove Invalid PubMed Text Rows

This section removes rows with missing or empty text values.

The PubMed corpus must contain valid biomedical text before being converted into sentence-level data. Empty text rows would produce invalid sentence and token sequences, so they are removed before further analysis.

In [ ]:
pubmed_df = pubmed_df.dropna(subset=["text"]).copy()

pubmed_df["text"] = pubmed_df["text"].astype(str).str.strip()
pubmed_df = pubmed_df[pubmed_df["text"] != ""].copy()

print("PubMed dataset shape after removing invalid text rows:", pubmed_df.shape)

## Check Duplicate PubMed Articles

This section checks duplicate PubMed articles using the `pmid` column.

Since PMID is a unique identifier for PubMed articles, duplicate PMID values indicate repeated records. Removing duplicates prevents the same document from being counted multiple times during EDA, preprocessing, pseudo-labeling, model training, and knowledge graph construction.

In [ ]:
duplicate_pmid_count = pubmed_df.duplicated(subset=["pmid"]).sum()

print("Duplicate PMID count:", duplicate_pmid_count)

pubmed_df = pubmed_df.drop_duplicates(subset=["pmid"]).copy()

print("PubMed dataset shape after duplicate removal:", pubmed_df.shape)

## Analyze Publication Year Distribution

This section analyzes the distribution of PubMed articles by publication year.

The publication year distribution helps verify whether the scraped corpus covers the intended time period and whether the dataset is heavily concentrated in specific years.

In [ ]:
year_counts = pubmed_df["publication_year"].value_counts().sort_index()

print("Publication year distribution:")
print(year_counts)

year_counts.plot(kind="bar", figsize=(8, 4))
plt.title("PubMed Article Distribution by Publication Year")
plt.xlabel("Publication Year")
plt.ylabel("Number of Articles")
plt.tight_layout()
plt.show()

## Analyze PubMed Text Length

This section calculates the word count of each combined PubMed text.

The combined `text` column contains the article title and abstract. Text length analysis helps identify very short or extremely long records that may affect downstream sentence splitting, tokenization, and NER preprocessing.

In [ ]:
pubmed_df["text_word_count"] = pubmed_df["text"].astype(str).str.split().str.len()

print("Text word count summary:")
print(pubmed_df["text_word_count"].describe())

pubmed_df[["pmid", "publication_year", "title", "text_word_count"]].sort_values(
    by="text_word_count"
).head(10)

## Visualize PubMed Text Length Distribution

This section visualizes the distribution of text lengths in the PubMed corpus.

Most biomedical abstracts are expected to have moderate length. Very short texts may contain limited context, while extremely long texts may require special attention during tokenization and sentence segmentation.

In [ ]:
pubmed_df["text_word_count"].plot(kind="hist", bins=50, figsize=(8, 4))

plt.title("Distribution of PubMed Text Word Counts")
plt.xlabel("Word Count")
plt.ylabel("Number of Articles")
plt.tight_layout()
plt.show()

## Analyze Biomedical Keyword Coverage

This section checks whether the PubMed corpus contains biomedical keywords that are relevant to the project scope.

The keywords are grouped into four broad categories aligned with the target entity types and downstream objectives:

1. Virus-related terms
2. Gene/protein-related terms
3. Disease-related terms
4. Drug/treatment-related terms

This analysis does not create labels. It is only used as an exploratory check to understand whether the scraped corpus contains enough topic coverage for downstream NER, knowledge graph construction, and drug repurposing analysis.

In [ ]:
keyword_groups = {
    "virus_related": [
        "virus",
        "viral",
        "SARS-CoV-2",
        "COVID-19",
        "influenza",
        "HIV",
        "hepatitis",
        "dengue"
    ],
    "gene_protein_related": [
        "gene",
        "protein",
        "ACE2",
        "TMPRSS2",
        "interferon",
        "cytokine",
        "receptor"
    ],
    "disease_related": [
        "disease",
        "infection",
        "syndrome",
        "cancer",
        "inflammation",
        "pneumonia"
    ],
    "drug_treatment_related": [
        "drug",
        "therapy",
        "treatment",
        "antiviral",
        "inhibitor",
        "vaccine"
    ]
}


def count_keyword_matches(text_series, keywords):
    # This function counts how many documents contain at least one keyword from a keyword group
    text_series_lower = text_series.astype(str).str.lower()
    keyword_pattern = "|".join(re.escape(keyword.lower()) for keyword in keywords)
    
    return text_series_lower.str.contains(keyword_pattern, regex=True).sum()


keyword_coverage_rows = []

for group_name, keywords in keyword_groups.items():
    matched_documents = count_keyword_matches(pubmed_df["text"], keywords)
    
    keyword_coverage_rows.append({
        "keyword_group": group_name,
        "matched_documents": matched_documents,
        "coverage_percentage": matched_documents / len(pubmed_df) * 100
    })

keyword_coverage_df = pd.DataFrame(keyword_coverage_rows)

keyword_coverage_df

## Visualize Biomedical Keyword Coverage

This section visualizes how many PubMed documents contain at least one keyword from each biomedical topic group.

The result helps confirm that the scraped corpus is not only virus-focused, but also contains enough terms related to genes, diseases, drugs, and treatments for the downstream biomedical NER and knowledge graph pipeline.

In [ ]:
keyword_coverage_df.plot(
    x="keyword_group",
    y="matched_documents",
    kind="bar",
    figsize=(8, 4),
    legend=False
)

plt.title("Biomedical Keyword Coverage in PubMed Corpus")
plt.xlabel("Keyword Group")
plt.ylabel("Number of Matched Documents")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Analyze Seed Entity Coverage

This section checks the presence of selected seed biomedical entities in the PubMed corpus.

The seed terms are not used as final labels. They are only used for EDA to verify whether the corpus contains examples related to the target entity types:

1. `Virus`
2. `Gene`
3. `Disease`
4. `Chemical` or drug-related terms

This step provides an early signal that the scraped corpus is relevant for distant supervision and downstream entity extraction.

In [ ]:
seed_entities = {
    "Virus": [
        "SARS-CoV-2",
        "HIV",
        "influenza",
        "HBV",
        "HCV",
        "DENV",
        "ZIKV"
    ],
    "Gene": [
        "ACE2",
        "TMPRSS2",
        "IL-6",
        "TNF",
        "IFN",
        "CD4",
        "CCR5"
    ],
    "Disease": [
        "COVID-19",
        "infection",
        "pneumonia",
        "cancer",
        "inflammation",
        "hepatitis"
    ],
    "Chemical": [
        "remdesivir",
        "ritonavir",
        "chloroquine",
        "antiviral",
        "vaccine",
        "inhibitor"
    ]
}


def count_seed_entity_occurrences(text_series, seed_terms):
    # This function counts document-level matches for each seed entity term
    text_series_lower = text_series.astype(str).str.lower()
    rows = []
    
    for term in seed_terms:
        matched_documents = text_series_lower.str.contains(
            re.escape(term.lower()),
            regex=True
        ).sum()
        
        rows.append({
            "term": term,
            "matched_documents": matched_documents
        })
    
    return rows


seed_coverage_rows = []

for entity_type, terms in seed_entities.items():
    term_rows = count_seed_entity_occurrences(pubmed_df["text"], terms)
    
    for row in term_rows:
        row["entity_type"] = entity_type
        seed_coverage_rows.append(row)

seed_coverage_df = pd.DataFrame(seed_coverage_rows)

seed_coverage_df.sort_values(
    by=["entity_type", "matched_documents"],
    ascending=[True, False]
)

## EDA Interpretation for PubMed Corpus

The previous EDA steps help validate that the scraped PubMed corpus is relevant for the project scope.

At this point, the dataset has been checked for:

1. Required columns
2. Missing text values
3. Duplicate PubMed IDs
4. Publication year distribution
5. Text length distribution
6. Biomedical keyword coverage
7. Seed entity coverage

The next step is to convert the document-level PubMed corpus into a sentence-level token dataset. This format is required for NER preprocessing because BIO labels are assigned at the token level.

## Define Sentence Splitting Function

This section defines a simple sentence splitting function for the PubMed text corpus.

The PubMed corpus is originally stored at the document level, where each row contains a full title and abstract. For NER preprocessing, the text needs to be converted into sentence-level rows because token-level BIO labels are usually assigned within sentence boundaries.

This function uses a regular-expression-based sentence splitter as a lightweight preprocessing approach.

In [ ]:
def split_into_sentences(text):
    # This function splits a document-level text into sentence-like units
    text = str(text).strip()
    
    if text == "":
        return []
    
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    
    return sentences

## Define Biomedical Tokenizer

This section defines a custom tokenizer for biomedical text.

A standard whitespace tokenizer can break biomedical entities incorrectly. For example, terms such as `SARS-CoV-2`, `HIV-1`, `COVID-19`, `IL-6`, and `ACE2` should be preserved as meaningful biomedical tokens.

The tokenizer is designed to keep alphanumeric biomedical terms with hyphens, slashes, and underscores as single tokens whenever possible.

In [ ]:
def biomedical_tokenize(sentence):
    # This function tokenizes biomedical sentences while preserving biomedical terms
    token_pattern = r"""
        [A-Za-z0-9]+(?:[-_/][A-Za-z0-9]+)*
        | [α-ωΑ-Ω]+
        | [^\w\s]
    """
    
    tokens = re.findall(token_pattern, str(sentence), flags=re.VERBOSE)
    
    return tokens

## Convert PubMed Documents into Sentence-Level Dataset

This section converts the PubMed corpus from document-level format into sentence-level format.

Each PubMed article is split into sentences. Each sentence is then tokenized using the biomedical tokenizer. Since the PubMed scraped corpus does not contain manual NER annotations, each token is initially assigned the `O` label.

The resulting dataset follows a token-level NER structure and will later be enriched with pseudo-labels for `Gene` and `Virus` through distant supervision.

In [ ]:
def create_pubmed_sentence_dataset(df):
    # This function converts document-level PubMed records into sentence-level NER rows
    sentence_rows = []
    
    for _, row in df.iterrows():
        sentences = split_into_sentences(row["text"])
        
        for sentence_id, sentence_text in enumerate(sentences):
            tokens = biomedical_tokenize(sentence_text)
            
            if len(tokens) < MIN_PUBMED_SENTENCE_TOKENS:
                continue
            
            decoded_tags = ["O"] * len(tokens)
            
            sentence_rows.append({
                "pmid": row["pmid"],
                "sentence_id": sentence_id,
                "sentence_text": sentence_text,
                "tokens": tokens,
                "decoded_tags": decoded_tags,
                "num_tokens": len(tokens),
                "publication_year": row.get("publication_year", None),
                "journal": row.get("journal", None),
                "url": row.get("url", None)
            })
    
    return pd.DataFrame(sentence_rows)

## Create PubMed Sentence-Level DataFrame

This cell applies the sentence-level preprocessing function to the PubMed corpus.

The output is a DataFrame where each row represents one sentence. The `tokens` column stores the biomedical token sequence, and the `decoded_tags` column stores the initial all-`O` BIO tag sequence.

This structure makes the PubMed corpus compatible with the downstream distant supervision and NER training pipeline.

In [ ]:
pubmed_sentences_df = create_pubmed_sentence_dataset(pubmed_df)

print("PubMed sentence-level dataset shape:", pubmed_sentences_df.shape)
pubmed_sentences_df.head()

## Inspect PubMed Sentence-Level Dataset

This section inspects the sentence-level PubMed dataset after preprocessing.

The goal is to verify that the document-level corpus has been successfully converted into NER-compatible rows, where each row contains a sentence, token sequence, and initial BIO label sequence.

In [ ]:
print("Sentence-level dataset shape:", pubmed_sentences_df.shape)

print("\nColumns:")
print(pubmed_sentences_df.columns.tolist())

print("\nToken count summary:")
print(pubmed_sentences_df["num_tokens"].describe())

## Validate Token and Label Alignment

This section checks whether every token sequence has the same length as its corresponding BIO label sequence.

This validation is essential for NER training because each token must have exactly one label. Any mismatch between `tokens` and `decoded_tags` would make the dataset invalid for token classification.

In [ ]:
pubmed_sentences_df["token_label_length_match"] = pubmed_sentences_df.apply(
    lambda row: len(row["tokens"]) == len(row["decoded_tags"]),
    axis=1
)

invalid_alignment_count = (~pubmed_sentences_df["token_label_length_match"]).sum()

print("Invalid token-label alignment count:", invalid_alignment_count)

if invalid_alignment_count > 0:
    raise ValueError("Some PubMed rows have mismatched token and label lengths.")

pubmed_sentences_df = pubmed_sentences_df.drop(columns=["token_label_length_match"])

## Visualize PubMed Sentence Length Distribution

This section visualizes the token length distribution of the sentence-level PubMed dataset.

Sentence length distribution is useful for downstream NER modeling because it helps determine whether sentence lengths are reasonable and whether extremely long sequences may require truncation during model training.

In [ ]:
pubmed_sentences_df["num_tokens"].plot(kind="hist", bins=50, figsize=(8, 4))

plt.title("Distribution of PubMed Sentence Token Counts")
plt.xlabel("Number of Tokens")
plt.ylabel("Number of Sentences")
plt.tight_layout()
plt.show()

## Save Preprocessed PubMed Dataset

This section saves the preprocessed PubMed sentence-level dataset.

The JSONL format is used as the main output because it preserves list-like columns such as `tokens` and `decoded_tags` more safely than CSV. A CSV version is also saved for quick inspection.

In [ ]:
pubmed_sentences_df.to_json(
    PUBMED_SENTENCE_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

pubmed_sentences_df.to_csv(
    PUBMED_SENTENCE_CSV_PATH,
    index=False
)

print("Saved PubMed JSONL:", PUBMED_SENTENCE_JSONL_PATH)
print("Saved PubMed CSV:", PUBMED_SENTENCE_CSV_PATH)

## PubMed Preprocessing Summary

At this stage, the PubMed scraped corpus has been converted from document-level data into a sentence-level NER-compatible dataset.

Each sentence-level row contains:

1. PubMed article metadata
2. Sentence text
3. Biomedical token sequence
4. Initial all-`O` BIO labels
5. Token count

Since PubMed does not contain manual NER annotations in this project, the initial labels are all `O`. The dataset will later be enriched using distant supervision for `Gene` and `Virus`.

# BC5CDR Dataset

This section starts the preprocessing pipeline for the BC5CDR dataset.

BC5CDR is used as the gold-labeled biomedical NER dataset in this project. Unlike the PubMed scraped corpus, BC5CDR already contains token-level annotations for two entity types:

1. `Chemical`
2. `Disease`

These gold labels will later be preserved and extended with pseudo-labels for `Gene` and `Virus` in the distant supervision stage.

## Load BC5CDR Dataset

This section loads the BC5CDR dataset from Hugging Face.

The dataset is loaded from the `tner/bc5cdr` source because it provides a token-level NER format that is suitable for this project. The dataset includes separate splits for training, validation, and testing.

In [ ]:
bc5cdr_dataset = load_dataset("tner/bc5cdr")

bc5cdr_dataset

## Define BC5CDR Label Mapping

This section defines the mapping from numeric label IDs to readable BIO labels.

The BC5CDR dataset stores NER labels as numeric IDs. To make the dataset easier to inspect and process, these numeric IDs are converted into readable BIO tags such as `B-Chemical`, `I-Chemical`, `B-Disease`, and `I-Disease`.

The readable labels will be stored in a new column called `decoded_tags`.

In [ ]:
bc5cdr_label_names = bc5cdr_dataset["train"].features["tags"].feature.names

id_to_label = {
    idx: label
    for idx, label in enumerate(bc5cdr_label_names)
}

label_to_id = {
    label: idx
    for idx, label in id_to_label.items()
}

print("BC5CDR label mapping:")
print(id_to_label)

## Convert BC5CDR Splits to DataFrames

This section converts each BC5CDR split into a pandas DataFrame.

For each row, the numeric tag sequence is converted into a readable BIO tag sequence and stored in the `decoded_tags` column. A `num_tokens` column is also added to support token length analysis and preprocessing decisions.

In [ ]:
def convert_bc5cdr_split_to_dataframe(dataset_split, split_name):
    # This function converts one BC5CDR split into a pandas DataFrame
    df = pd.DataFrame(dataset_split)
    
    df["split"] = split_name
    df["decoded_tags"] = df["tags"].apply(
        lambda tag_ids: [id_to_label[tag_id] for tag_id in tag_ids]
    )
    df["num_tokens"] = df["tokens"].apply(len)
    
    return df


bc5cdr_dfs = {}

for split_name in ["train", "validation", "test"]:
    bc5cdr_dfs[split_name] = convert_bc5cdr_split_to_dataframe(
        dataset_split=bc5cdr_dataset[split_name],
        split_name=split_name
    )
    
    print(split_name, bc5cdr_dfs[split_name].shape)

## Inspect BC5CDR Split Sizes

This section checks the number of rows in each BC5CDR split.

Understanding the split size is important before preprocessing because train, validation, and test sets must be handled consistently while still preserving their original separation.

In [ ]:
bc5cdr_split_summary = []

for split_name, df in bc5cdr_dfs.items():
    bc5cdr_split_summary.append({
        "split": split_name,
        "num_sentences": len(df),
        "avg_tokens": df["num_tokens"].mean(),
        "min_tokens": df["num_tokens"].min(),
        "max_tokens": df["num_tokens"].max()
    })

bc5cdr_split_summary_df = pd.DataFrame(bc5cdr_split_summary)

bc5cdr_split_summary_df

## Analyze BC5CDR Token Length Distribution

This section analyzes the token length distribution of BC5CDR sentences.

Token length analysis helps identify very short sentences, very long sentences, and general sequence length patterns. This information is useful for preprocessing decisions and later NER model training.

In [ ]:
for split_name, df in bc5cdr_dfs.items():
    print(f"\n{split_name.upper()} token count summary:")
    print(df["num_tokens"].describe())

## Visualize BC5CDR Token Length Distribution

This section visualizes the sentence token length distribution for each BC5CDR split.

The visualization helps compare whether train, validation, and test splits have similar sentence length patterns.

In [ ]:
for split_name, df in bc5cdr_dfs.items():
    df["num_tokens"].plot(kind="hist", bins=50, figsize=(8, 4))
    
    plt.title(f"BC5CDR {split_name.capitalize()} Token Count Distribution")
    plt.xlabel("Number of Tokens")
    plt.ylabel("Number of Sentences")
    plt.tight_layout()
    plt.show()

## Analyze BC5CDR Label Distribution

This section counts the token-level BIO label distribution in each BC5CDR split.

Label distribution helps verify how many tokens are labeled as `Chemical`, `Disease`, or `O`. This is important because biomedical NER datasets are usually highly imbalanced, with most tokens labeled as `O`.

In [ ]:
def count_bio_labels(tag_sequences):
    # This function counts BIO labels from a list of tag sequences
    label_counter = Counter()
    
    for tags in tag_sequences:
        label_counter.update(tags)
    
    return label_counter


bc5cdr_label_distribution_rows = []

for split_name, df in bc5cdr_dfs.items():
    label_counter = count_bio_labels(df["decoded_tags"])
    
    for label, count in label_counter.items():
        bc5cdr_label_distribution_rows.append({
            "split": split_name,
            "label": label,
            "count": count
        })

bc5cdr_label_distribution_df = pd.DataFrame(bc5cdr_label_distribution_rows)

bc5cdr_label_distribution_df.sort_values(
    by=["split", "label"]
)

## Visualize BC5CDR Label Distribution

This section visualizes the token-level label distribution for each BC5CDR split.

This visualization provides a quick view of class imbalance between entity labels and the non-entity `O` label.

In [ ]:
for split_name in ["train", "validation", "test"]:
    split_label_df = bc5cdr_label_distribution_df[
        bc5cdr_label_distribution_df["split"] == split_name
    ].copy()
    
    split_label_df.plot(
        x="label",
        y="count",
        kind="bar",
        figsize=(8, 4),
        legend=False
    )
    
    plt.title(f"BC5CDR {split_name.capitalize()} Label Distribution")
    plt.xlabel("BIO Label")
    plt.ylabel("Token Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Extract Entity Mentions from BIO Tags

This section defines a helper function to extract complete entity mentions from BIO tag sequences.

Token-level label counts are useful, but entity-level analysis gives a clearer understanding of how many biomedical mentions appear in the dataset. For example, a multi-token entity such as `breast cancer` should be counted as one `Disease` mention rather than two separate labeled tokens.

The extracted entity mentions will be used to analyze Chemical and Disease coverage in BC5CDR.

In [ ]:
def extract_entities_from_bio(tokens, tags):
    # This function extracts entity spans from tokens and BIO tags
    entities = []
    current_entity = None
    
    for idx, (token, tag) in enumerate(zip(tokens, tags)):
        if tag == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
            continue
        
        if "-" not in tag:
            continue
        
        prefix, entity_type = tag.split("-", 1)
        
        if prefix == "B":
            if current_entity is not None:
                entities.append(current_entity)
            
            current_entity = {
                "entity_type": entity_type,
                "start_token": idx,
                "end_token": idx + 1,
                "tokens": [token]
            }
        
        elif prefix == "I":
            if current_entity is not None and current_entity["entity_type"] == entity_type:
                current_entity["end_token"] = idx + 1
                current_entity["tokens"].append(token)
            else:
                current_entity = {
                    "entity_type": entity_type,
                    "start_token": idx,
                    "end_token": idx + 1,
                    "tokens": [token]
                }
    
    if current_entity is not None:
        entities.append(current_entity)
    
    for entity in entities:
        entity["entity_text"] = " ".join(entity["tokens"])
    
    return entities

## Build BC5CDR Entity-Level DataFrame

This section converts BC5CDR token-level BIO annotations into an entity-level DataFrame.

Each row in the resulting DataFrame represents one complete entity mention. This format makes it easier to count entity mentions, inspect top entities, and compare Chemical and Disease coverage across dataset splits.

In [ ]:
def build_bc5cdr_entity_dataframe(bc5cdr_dfs):
    # This function builds an entity-level DataFrame from all BC5CDR splits
    entity_rows = []
    
    for split_name, df in bc5cdr_dfs.items():
        for row_idx, row in df.iterrows():
            entities = extract_entities_from_bio(
                tokens=row["tokens"],
                tags=row["decoded_tags"]
            )
            
            for entity in entities:
                entity_rows.append({
                    "split": split_name,
                    "row_idx": row_idx,
                    "entity_type": entity["entity_type"],
                    "entity_text": entity["entity_text"],
                    "start_token": entity["start_token"],
                    "end_token": entity["end_token"],
                    "tokens": entity["tokens"]
                })
    
    return pd.DataFrame(entity_rows)


bc5cdr_entities_df = build_bc5cdr_entity_dataframe(bc5cdr_dfs)

print("BC5CDR entity-level DataFrame shape:", bc5cdr_entities_df.shape)
bc5cdr_entities_df.head()

## Analyze BC5CDR Entity Distribution

This section analyzes the number of entity mentions by split and entity type.

Unlike token-level label distribution, this analysis counts complete entity mentions. It helps verify whether both `Chemical` and `Disease` entities are available across train, validation, and test splits.

In [ ]:
bc5cdr_entity_distribution_df = (
    bc5cdr_entities_df
    .groupby(["split", "entity_type"])
    .size()
    .reset_index(name="entity_count")
)

bc5cdr_entity_distribution_df

## Inspect Top BC5CDR Entity Mentions

This section shows the most frequent entity mentions in BC5CDR.

Top entity inspection is useful for understanding the biomedical domain coverage of the dataset and for identifying common chemicals or diseases that appear repeatedly in the corpus.

In [ ]:
top_bc5cdr_entities_df = (
    bc5cdr_entities_df
    .groupby(["entity_type", "entity_text"])
    .size()
    .reset_index(name="mention_count")
    .sort_values(by=["entity_type", "mention_count"], ascending=[True, False])
)

top_bc5cdr_entities_df.groupby("entity_type").head(20)

## Identify Short Non-Informative BC5CDR Sentences

This section identifies very short BC5CDR sentences that contain no entity labels.

Short sentences with only `O` labels usually provide limited learning signal for NER training. However, short sentences that contain entity labels should be preserved because they still contain useful biomedical annotations.

The filtering rule is therefore conservative:

- Remove short sentences only if they contain no entity labels.
- Keep short sentences if they contain at least one `Chemical` or `Disease` label.

In [ ]:
def has_entity_label(tags):
    # This function checks whether a BIO tag sequence contains at least one entity label
    return any(tag != "O" for tag in tags)


bc5cdr_short_sentence_summary = []

for split_name, df in bc5cdr_dfs.items():
    short_sentence_mask = df["num_tokens"] <= MIN_BC5CDR_SHORT_SENTENCE_TOKENS
    no_entity_mask = ~df["decoded_tags"].apply(has_entity_label)
    
    short_no_entity_count = (short_sentence_mask & no_entity_mask).sum()
    short_with_entity_count = (short_sentence_mask & ~no_entity_mask).sum()
    
    bc5cdr_short_sentence_summary.append({
        "split": split_name,
        "short_no_entity_sentences": short_no_entity_count,
        "short_with_entity_sentences": short_with_entity_count
    })

bc5cdr_short_sentence_summary_df = pd.DataFrame(bc5cdr_short_sentence_summary)

bc5cdr_short_sentence_summary_df

## Remove Short Non-Informative BC5CDR Sentences

This section applies the conservative filtering rule to BC5CDR.

Only short sentences with no entity labels are removed. This reduces non-informative all-`O` examples while preserving short sentences that contain valid Chemical or Disease annotations.

In [ ]:
bc5cdr_dfs_clean = {}

for split_name, df in bc5cdr_dfs.items():
    short_sentence_mask = df["num_tokens"] <= MIN_BC5CDR_SHORT_SENTENCE_TOKENS
    no_entity_mask = ~df["decoded_tags"].apply(has_entity_label)
    
    keep_mask = ~(short_sentence_mask & no_entity_mask)
    
    df_clean = df[keep_mask].copy()
    bc5cdr_dfs_clean[split_name] = df_clean
    
    print(split_name)
    print("Before:", df.shape)
    print("After :", df_clean.shape)

## Validate BC5CDR Token and Label Alignment

This section validates whether every BC5CDR token sequence has the same length as its corresponding BIO tag sequence.

This validation is required before saving the processed dataset because NER training expects one label for each token.

In [ ]:
for split_name, df in bc5cdr_dfs_clean.items():
    alignment_matches = df.apply(
        lambda row: len(row["tokens"]) == len(row["decoded_tags"]),
        axis=1
    )
    
    invalid_alignment_count = (~alignment_matches).sum()
    
    print(split_name, "invalid token-label alignment count:", invalid_alignment_count)
    
    if invalid_alignment_count > 0:
        raise ValueError(f"Invalid token-label alignment found in {split_name}.")

## Save Preprocessed BC5CDR Splits

This section saves the cleaned BC5CDR splits into JSONL files.

Each split is saved separately to preserve the original train, validation, and test separation. The JSONL format is used because it preserves list-like columns such as `tokens` and `decoded_tags`.

These files will later be used in the distant supervision stage, where `Gene` and `Virus` pseudo-labels will be added without overwriting the original Chemical and Disease gold labels.

In [ ]:
bc5cdr_output_paths = {}

for split_name, df in bc5cdr_dfs_clean.items():
    output_path = os.path.join(
        BC5CDR_OUTPUT_DIR,
        f"bc5cdr_{split_name}_preprocessed.jsonl"
    )
    
    df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False
    )
    
    bc5cdr_output_paths[split_name] = output_path
    
    print(f"Saved {split_name} split:", output_path)

## BC5CDR Preprocessing Summary

At this stage, the BC5CDR dataset has been loaded, decoded, inspected, lightly cleaned, validated, and saved.

The original Chemical and Disease annotations are preserved as gold labels. The preprocessing only removes short non-informative all-`O` sentences while keeping any sentence that contains biomedical entity annotations.

The preprocessed BC5CDR splits will be used in the next stage for distant supervision. In that stage, `Gene` and `Virus` pseudo-labels will be added only to tokens that are still labeled as `O`.

# Final Notebook Summary

This notebook completed the EDA and preprocessing stage for both PubMed and BC5CDR.

For the PubMed scraped corpus, the notebook:

1. Loaded the cleaned document-level PubMed dataset.
2. Inspected missing values, duplicates, publication years, and text lengths.
3. Checked biomedical keyword and seed entity coverage.
4. Split document-level text into sentence-level rows.
5. Tokenized sentences using a biomedical tokenizer.
6. Assigned initial all-`O` BIO labels.
7. Saved the sentence-level PubMed dataset.

For BC5CDR, the notebook:

1. Loaded the dataset from Hugging Face.
2. Decoded numeric NER labels into readable BIO tags.
3. Analyzed split sizes, token lengths, label distributions, and entity mentions.
4. Removed short non-informative all-`O` sentences.
5. Validated token-label alignment.
6. Saved the cleaned train, validation, and test splits.

The outputs from this notebook provide the standardized input datasets for the next project stage: distant supervision using HGNC and NCBI Taxonomy dictionaries.